# Reinforcement Learning — From Basics to Deep Q-Networks

- **RL Fundamentals**: Agent, Environment, State, Action, Reward
- **Q-Learning**: Tabular value-based method
- **Deep Q-Network (DQN)**: Neural network approximates Q-values
- **Policy Gradient** introduction
- **OpenAI Gym** environments

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random

try:
    import gymnasium as gym
    print(f'Gymnasium: {gym.__version__}')
except ImportError:
    try:
        import gym
        print(f'OpenAI Gym: {gym.__version__}')
    except ImportError:
        print('Install: pip install gymnasium')

import warnings
warnings.filterwarnings('ignore')

## 1. RL Concepts

```
┌─────────────────────────────────────────┐
│                                         │
│   Agent ──action──→ Environment         │
│     ↑                   │               │
│     └──reward, state────┘               │
│                                         │
└─────────────────────────────────────────┘
```

**Key Terms:**
- **State (s)**: Current situation
- **Action (a)**: What the agent does
- **Reward (r)**: Feedback signal
- **Policy (π)**: Strategy — maps states to actions
- **Q-value Q(s,a)**: Expected future reward for taking action a in state s
- **Discount factor (γ)**: How much to value future vs immediate rewards

**Goal**: Learn a policy that maximizes cumulative reward

## 2. Custom Grid World — Q-Learning from Scratch

In [ ]:
class GridWorld:
    """Simple 4x4 grid world.
    Agent starts at (0,0), goal at (3,3), obstacles at (1,1), (2,2).
    """
    def __init__(self):
        self.grid_size = 4
        self.start = (0, 0)
        self.goal = (3, 3)
        self.obstacles = [(1, 1), (2, 2)]
        self.state = self.start
        # Actions: 0=up, 1=right, 2=down, 3=left
        self.actions = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}
        self.action_names = ['↑', '→', '↓', '←']
    
    def reset(self):
        self.state = self.start
        return self._state_to_idx(self.state)
    
    def _state_to_idx(self, state):
        return state[0] * self.grid_size + state[1]
    
    def step(self, action):
        dr, dc = self.actions[action]
        new_r = max(0, min(self.grid_size - 1, self.state[0] + dr))
        new_c = max(0, min(self.grid_size - 1, self.state[1] + dc))
        new_state = (new_r, new_c)
        
        if new_state in self.obstacles:
            reward = -10
            done = False
            # Stay in place
        elif new_state == self.goal:
            reward = 100
            done = True
            self.state = new_state
        else:
            reward = -1
            done = False
            self.state = new_state
        
        return self._state_to_idx(self.state), reward, done

env = GridWorld()
n_states = env.grid_size ** 2
n_actions = 4
print(f'Grid: {env.grid_size}x{env.grid_size} | States: {n_states} | Actions: {n_actions}')

In [ ]:
# Q-Learning
Q_table = np.zeros((n_states, n_actions))

# Hyperparameters
alpha = 0.1        # Learning rate
gamma = 0.99       # Discount factor
epsilon = 1.0      # Exploration rate
epsilon_min = 0.01
epsilon_decay = 0.995
episodes = 1000

rewards_per_episode = []

for ep in range(episodes):
    state = env.reset()
    total_reward = 0
    done = False
    steps = 0
    
    while not done and steps < 100:
        # Epsilon-greedy action selection
        if np.random.random() < epsilon:
            action = np.random.randint(n_actions)  # Explore
        else:
            action = np.argmax(Q_table[state])  # Exploit
        
        next_state, reward, done = env.step(action)
        
        # Q-Learning update: Q(s,a) ← Q(s,a) + α[r + γ·max Q(s',a') - Q(s,a)]
        Q_table[state, action] += alpha * (
            reward + gamma * np.max(Q_table[next_state]) - Q_table[state, action]
        )
        
        state = next_state
        total_reward += reward
        steps += 1
    
    rewards_per_episode.append(total_reward)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

# Plot learning progress
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
window = 50
smoothed = np.convolve(rewards_per_episode, np.ones(window)/window, mode='valid')
axes[0].plot(smoothed)
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward (smoothed)')
axes[0].set_title('Q-Learning Training Progress'); axes[0].grid(True, alpha=0.3)

# Visualize Q-table as policy
policy = np.argmax(Q_table, axis=1).reshape(4, 4)
policy_arrows = np.array([env.action_names[a] for a in policy.flatten()]).reshape(4, 4)

axes[1].imshow(np.max(Q_table, axis=1).reshape(4, 4), cmap='YlOrRd')
for i in range(4):
    for j in range(4):
        if (i, j) == env.goal:
            axes[1].text(j, i, '★', ha='center', va='center', fontsize=20)
        elif (i, j) in env.obstacles:
            axes[1].text(j, i, '■', ha='center', va='center', fontsize=20)
        else:
            axes[1].text(j, i, policy_arrows[i, j], ha='center', va='center', fontsize=16)
axes[1].set_title('Learned Policy (arrows = actions)')
plt.tight_layout()
plt.show()

## 3. Deep Q-Network (DQN) Concept

When state space is too large for a table → use a **neural network** to approximate Q-values.

```
State → [Neural Network] → Q(s, a₁), Q(s, a₂), ..., Q(s, aₙ)
```

Key DQN innovations:
1. **Experience Replay**: Store transitions, sample random mini-batches
2. **Target Network**: Separate network for stable Q-value targets
3. **ε-greedy**: Explore randomly with decreasing probability

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001
        self.model = self._build_model()
        self.target_model = self._build_model()
        self.update_target_model()
    
    def _build_model(self):
        model = keras.Sequential([
            layers.Dense(64, activation='relu', input_dim=self.state_size),
            layers.Dense(64, activation='relu'),
            layers.Dense(self.action_size, activation='linear')
        ])
        model.compile(loss='mse', optimizer=keras.optimizers.Adam(self.learning_rate))
        return model
    
    def update_target_model(self):
        self.target_model.set_weights(self.model.get_weights())
    
    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))
    
    def act(self, state):
        if np.random.random() <= self.epsilon:
            return random.randrange(self.action_size)
        q_values = self.model.predict(state, verbose=0)
        return np.argmax(q_values[0])
    
    def replay(self, batch_size=32):
        if len(self.memory) < batch_size:
            return
        minibatch = random.sample(self.memory, batch_size)
        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                target += self.gamma * np.amax(self.target_model.predict(next_state, verbose=0)[0])
            target_f = self.model.predict(state, verbose=0)
            target_f[0][action] = target
            self.model.fit(state, target_f, epochs=1, verbose=0)
        
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

print('DQN Agent architecture:')
agent = DQNAgent(state_size=4, action_size=2)
agent.model.summary()

## 4. RL Algorithm Taxonomy

```
RL Algorithms
├── Value-Based (learn Q-values)
│   ├── Q-Learning (tabular)
│   ├── DQN (Deep Q-Network)
│   └── Double DQN, Dueling DQN
├── Policy-Based (learn policy directly)
│   ├── REINFORCE (Monte Carlo policy gradient)
│   └── PPO (Proximal Policy Optimization)
├── Actor-Critic (combine both)
│   ├── A2C / A3C
│   └── SAC (Soft Actor-Critic)
└── Model-Based (learn environment model)
    ├── Dyna-Q
    └── MuZero, Dreamer
```

### When to Use:
| Algorithm | Discrete Actions | Continuous Actions | Stability |
|-----------|:---:|:---:|:---:|
| **DQN** | ✅ | ❌ | Medium |
| **PPO** | ✅ | ✅ | **High** |
| **SAC** | ✅ | ✅ | High |
| **A3C** | ✅ | ✅ | Medium |

### Real-World Applications:
- **Game AI**: AlphaGo, OpenAI Five, Atari
- **Robotics**: Manipulation, locomotion
- **Autonomous Driving**: Lane keeping, decision making
- **Recommendation Systems**: Sequential recommendations
- **Resource Optimization**: Data center cooling (Google)